# SASRec Baseline On MASI CSJ

This notebook trains the exact SASRec-style sequential ID baseline on the same Amazon Reviews 2023 Clothing/Shoes/Jewelry split contract used by the MASI full-dataset notebook. It writes `HR@10`, `NDCG@10`, `Coverage@10`, and latency metrics to `outputs/baselines/sasrec_full_dataset/baseline_summary.json`.

Use `smoke_safe` for a fast check and `long_safe` to match the current bounded MASI comparison profile.

In [ ]:
from __future__ import annotations

from copy import deepcopy
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys


REPO_URL = "https://github.com/pradyunuydarp/MASI.git"
REPO_BRANCH = "main"
KAGGLE_WORKING_ROOT = Path("/kaggle/working")
KAGGLE_INPUT_ROOT = Path("/kaggle/input")
RUNNING_ON_KAGGLE = KAGGLE_WORKING_ROOT.exists() and KAGGLE_INPUT_ROOT.exists()
USE_GIT_CLONE_ON_KAGGLE = True
KAGGLE_REPO_DIR = KAGGLE_WORKING_ROOT / "MASI"
STORAGE_ROOT = KAGGLE_WORKING_ROOT / "masi_artifacts" if RUNNING_ON_KAGGLE else None
BASELINE_PROFILE = "long_safe"  # "smoke_safe" for a quick validation run
RUN_PIP_INSTALL = True

PROFILES = {
    "smoke_safe": {
        "dataset": {"max_users": 256, "max_items": 512, "max_review_records": 150000},
        "baseline": {"epochs": 2, "batch_size": 128, "max_eval_candidates": 512},
    },
    "long_safe": {
        "dataset": {"max_users": 12288, "max_items": 24576, "max_review_records": 50000000},
        "baseline": {"epochs": 30, "batch_size": 256, "max_eval_candidates": 1024},
    },
}


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "masi").exists():
            return candidate
    raise FileNotFoundError("Could not find the MASI repository root.")


if RUNNING_ON_KAGGLE and USE_GIT_CLONE_ON_KAGGLE:
    KAGGLE_WORKING_ROOT.mkdir(parents=True, exist_ok=True)
    os.chdir(KAGGLE_WORKING_ROOT)
    if KAGGLE_REPO_DIR.exists():
        shutil.rmtree(KAGGLE_REPO_DIR)
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(KAGGLE_REPO_DIR)],
        check=True,
        cwd=KAGGLE_WORKING_ROOT,
    )
    REPO_DIR = KAGGLE_REPO_DIR
else:
    REPO_DIR = find_repo_root(Path.cwd())

if STORAGE_ROOT is None:
    STORAGE_ROOT = REPO_DIR
STORAGE_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(REPO_DIR)
print(f"Repo: {REPO_DIR}")
print(f"Storage: {STORAGE_ROOT}")

In [ ]:
# Repair older GitHub clones that predate the baseline artifacts.
BASELINE_BOOTSTRAP_FILES = {
    'configs/baseline_sasrec_full_dataset.json': '{\n  "seed": 7,\n  "runtime": {\n    "device": "auto",\n    "log_device_summary": true,\n    "run_name": "baseline_sasrec_full_dataset"\n  },\n  "dataset": {\n    "reviews_path": "data/full_dataset/Clothing_Shoes_and_Jewelry.jsonl",\n    "kaggle_input_slugs": [\n      "masi-amazon-csj-full-dataset"\n    ],\n    "reviews_relpath": "Clothing_Shoes_and_Jewelry.jsonl",\n    "min_user_interactions": 5,\n    "min_item_interactions": 5,\n    "max_users": 12288,\n    "max_items": 24576,\n    "max_review_records": 50000000,\n    "review_record_offset": 0,\n    "user_rank_offset": 0,\n    "item_rank_offset": 0,\n    "collapse_consecutive_duplicates": false\n  },\n  "baseline": {\n    "name": "sasrec",\n    "max_sequence_length": 50,\n    "hidden_dim": 128,\n    "num_heads": 4,\n    "num_layers": 3,\n    "dropout": 0.1,\n    "batch_size": 256,\n    "learning_rate": 0.001,\n    "epochs": 30,\n    "top_k": 10,\n    "max_eval_candidates": 1024,\n    "cold_start_ratio": 0.2,\n    "min_train_history": 1,\n    "use_cold_start_evaluation": true\n  },\n  "outputs_root": "outputs/baselines/sasrec_full_dataset",\n  "checkpoint_root": "outputs/baselines/sasrec_full_dataset/checkpoints"\n}\n',
    'scripts/run_sasrec_baseline.py': '#!/usr/bin/env python3\n"""Run the SASRec baseline on the same CSJ split contract used by MASI."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport random\nfrom pathlib import Path\n\nimport torch\n\nfrom masi.baselines import run_sasrec_baseline\nfrom masi.common.config import find_repo_root, load_json_config\nfrom masi.common.io import ensure_directory\nfrom masi.common.runtime import (\n    detect_runtime_environment,\n    find_kaggle_dataset_root,\n    resolve_input_path,\n    resolve_storage_root,\n    resolve_torch_device,\n)\nfrom masi.recommender.amazon_data import build_real_amazon_histories\n\n\ndef parse_args() -> argparse.Namespace:\n    """Parse command-line arguments."""\n\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\n        "--config",\n        default="configs/baseline_sasrec_full_dataset.json",\n        help="Path to the SASRec baseline config.",\n    )\n    parser.add_argument(\n        "--storage-root",\n        default=None,\n        help="Optional storage root for data, checkpoints, and outputs.",\n    )\n    return parser.parse_args()\n\n\ndef _optional_positive_int(value: object) -> int | None:\n    """Convert loose config values into optional positive integers."""\n\n    if value is None:\n        return None\n    parsed = int(value)\n    return parsed if parsed > 0 else None\n\n\ndef main() -> None:\n    """Load MASI-format data, train SASRec, and write baseline metrics."""\n\n    args = parse_args()\n    loaded = load_json_config(args.config)\n    config = loaded.data\n    repo_root = find_repo_root(Path(__file__))\n    runtime_config = dict(config.get("runtime", {}))\n    storage_root = resolve_storage_root(\n        repo_root=repo_root,\n        runtime_config=runtime_config,\n        cli_storage_root=args.storage_root,\n    )\n\n    seed = int(config["seed"])\n    random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    device = resolve_torch_device(runtime_config)\n\n    dataset_config = dict(config["dataset"])\n    dataset_root = find_kaggle_dataset_root(\n        dataset_slugs=dataset_config.get("kaggle_input_slugs"),\n        required_relative_paths=[\n            dataset_config.get("reviews_relpath") or dataset_config.get("reviews_path"),\n        ],\n    )\n    reviews_path = resolve_input_path(\n        repo_root=repo_root,\n        storage_root=storage_root,\n        configured_path=str(dataset_config.get("reviews_path", "")),\n        kaggle_dataset_root=dataset_root,\n        relative_path=str(dataset_config.get("reviews_relpath", "")).strip() or None,\n    )\n    if reviews_path is None or not reviews_path.exists():\n        raise FileNotFoundError(\n            "Missing reviews input for SASRec baseline. "\n            f"Configured path: {dataset_config.get(\'reviews_path\')}"\n        )\n\n    imported = build_real_amazon_histories(\n        reviews_path=str(reviews_path),\n        min_user_interactions=int(dataset_config["min_user_interactions"]),\n        min_item_interactions=_optional_positive_int(dataset_config.get("min_item_interactions")),\n        max_users=int(dataset_config.get("max_users", 0) or 0),\n        max_items=int(dataset_config.get("max_items", 0) or 0),\n        max_review_records=_optional_positive_int(dataset_config.get("max_review_records")),\n        review_record_offset=int(dataset_config.get("review_record_offset", 0) or 0),\n        user_rank_offset=int(dataset_config.get("user_rank_offset", 0) or 0),\n        item_rank_offset=int(dataset_config.get("item_rank_offset", 0) or 0),\n        collapse_consecutive_duplicates=bool(dataset_config.get("collapse_consecutive_duplicates", False)),\n    )\n    imported.summary["reviews_path"] = str(reviews_path.resolve())\n    imported.summary["resolved_dataset_root"] = str(dataset_root.resolve()) if dataset_root is not None else None\n    imported.summary["environment"] = detect_runtime_environment()\n\n    run_name = str(runtime_config.get("run_name", "baseline_sasrec_full_dataset"))\n    outputs_root = config.get("outputs_root")\n    if outputs_root:\n        outputs_path = Path(str(outputs_root)).expanduser()\n        outputs_path = outputs_path if outputs_path.is_absolute() else storage_root / outputs_path\n    else:\n        outputs_path = storage_root / "outputs" / run_name\n    checkpoint_root = config.get("checkpoint_root")\n    checkpoint_path = None\n    if checkpoint_root:\n        raw_checkpoint_path = Path(str(checkpoint_root)).expanduser()\n        checkpoint_path = raw_checkpoint_path if raw_checkpoint_path.is_absolute() else storage_root / raw_checkpoint_path\n    else:\n        checkpoint_path = outputs_path / "checkpoints"\n\n    summary = run_sasrec_baseline(\n        user_histories=imported.user_histories,\n        import_summary=imported.summary,\n        config=config,\n        device=device,\n        outputs_root=ensure_directory(outputs_path),\n        checkpoint_root=checkpoint_path,\n    )\n    print(json.dumps(summary, indent=2))\n    print(f"Wrote SASRec baseline summary to {outputs_path / \'baseline_summary.json\'}")\n\n\nif __name__ == "__main__":\n    main()\n',
    'src/masi/baselines/__init__.py': '"""Baseline experiment helpers for MASI evaluation."""\n\nfrom masi.baselines.sasrec import run_sasrec_baseline\n\n__all__ = ["run_sasrec_baseline"]\n',
    'src/masi/baselines/sasrec.py': '"""SASRec baseline training and evaluation on MASI data splits."""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nimport hashlib\nimport time\nfrom random import Random\nfrom statistics import mean\nfrom pathlib import Path\nfrom typing import Iterable\n\nimport torch\nfrom torch import nn\nfrom torch.utils.data import DataLoader, Dataset\n\nfrom masi.common.io import ensure_directory, write_json\nfrom masi.common.progress import make_progress_bar\nfrom masi.common.runtime import module_state_dict_to_cpu\nfrom masi.recommender.evaluation import (\n    LeaveOneOutExample,\n    build_leave_one_out_split,\n    coverage_at_k,\n    hit_rate_at_k,\n    ndcg_at_k,\n)\nfrom masi.recommender.sasrec import SASRecConfig, SASRecModel\n\n\n@dataclass(slots=True)\nclass SASRecTrainingExample:\n    """One SASRec next-item training example."""\n\n    user_id: str\n    history_item_ids: list[str]\n    target_item_id: str\n    input_item_ids: list[int]\n    label_item_id: int\n\n\nclass SASRecTrainingDataset(Dataset[SASRecTrainingExample]):\n    """Convert chronological item histories into SASRec training examples."""\n\n    def __init__(\n        self,\n        *,\n        user_histories: dict[str, list[str]],\n        item_to_index: dict[str, int],\n        max_sequence_length: int,\n        pad_token_id: int = 0,\n    ) -> None:\n        self.examples: list[SASRecTrainingExample] = []\n        self.max_sequence_length = max_sequence_length\n        self.pad_token_id = pad_token_id\n\n        for user_id, history in sorted(user_histories.items()):\n            indexed_history = [item_to_index[item_id] for item_id in history if item_id in item_to_index]\n            if len(indexed_history) < 2:\n                continue\n\n            for prediction_index in range(1, len(indexed_history)):\n                raw_history = history[:prediction_index]\n                raw_target = history[prediction_index]\n                input_ids = _left_pad(\n                    indexed_history[:prediction_index],\n                    max_length=max_sequence_length,\n                    pad_token_id=pad_token_id,\n                )\n                self.examples.append(\n                    SASRecTrainingExample(\n                        user_id=user_id,\n                        history_item_ids=list(raw_history),\n                        target_item_id=raw_target,\n                        input_item_ids=input_ids,\n                        label_item_id=indexed_history[prediction_index],\n                    )\n                )\n\n    def __len__(self) -> int:\n        """Return the number of next-item training examples."""\n\n        return len(self.examples)\n\n    def __getitem__(self, index: int) -> SASRecTrainingExample:\n        """Return one SASRec training example."""\n\n        return self.examples[index]\n\n    @staticmethod\n    def collate(batch: list[SASRecTrainingExample]) -> dict[str, torch.Tensor]:\n        """Convert examples into tensors."""\n\n        return {\n            "item_sequences": torch.tensor([example.input_item_ids for example in batch], dtype=torch.long),\n            "labels": torch.tensor([example.label_item_id for example in batch], dtype=torch.long),\n        }\n\n\ndef _left_pad(sequence: Iterable[int], *, max_length: int, pad_token_id: int) -> list[int]:\n    """Left-pad or left-truncate a sequence so the last position is recent."""\n\n    values = list(sequence)[-max_length:]\n    return [pad_token_id] * (max_length - len(values)) + values\n\n\ndef _build_item_index(\n    *,\n    user_histories: dict[str, list[str]],\n    warm_examples: list[LeaveOneOutExample],\n    cold_examples: list[LeaveOneOutExample],\n) -> tuple[dict[str, int], dict[int, str]]:\n    """Build a stable item index with zero reserved for padding."""\n\n    item_ids = {\n        item_id\n        for history in user_histories.values()\n        for item_id in history\n    }\n    item_ids.update(example.target_item_id for example in warm_examples)\n    item_ids.update(example.target_item_id for example in cold_examples)\n\n    item_to_index = {item_id: index for index, item_id in enumerate(sorted(item_ids), start=1)}\n    index_to_item = {index: item_id for item_id, index in item_to_index.items()}\n    return item_to_index, index_to_item\n\n\ndef _candidate_pool_for_example(\n    *,\n    candidate_item_ids: list[str],\n    target_item_id: str,\n    max_eval_candidates: int | None,\n    seed: int,\n    user_id: str,\n) -> list[str]:\n    """Return a deterministic candidate subset that always includes the target."""\n\n    if max_eval_candidates is None or max_eval_candidates <= 0 or len(candidate_item_ids) <= max_eval_candidates:\n        return candidate_item_ids\n\n    cap = max(1, int(max_eval_candidates))\n    negatives = [item_id for item_id in candidate_item_ids if item_id != target_item_id]\n    digest = hashlib.sha256(f"{seed}:{user_id}:{target_item_id}".encode("utf-8")).hexdigest()\n    rng = Random(int(digest[:16], 16))\n    sampled_negative_count = max(0, cap - 1)\n    if len(negatives) > sampled_negative_count:\n        negatives = rng.sample(negatives, sampled_negative_count)\n    return sorted([target_item_id, *negatives])\n\n\ndef _train_sasrec(\n    *,\n    model: SASRecModel,\n    train_loader: DataLoader,\n    learning_rate: float,\n    epochs: int,\n    device: torch.device,\n    pad_token_id: int,\n) -> list[float]:\n    """Train SASRec with full-catalog next-item cross entropy."""\n\n    if len(train_loader) == 0 or epochs <= 0:\n        return []\n\n    model.to(device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)\n    loss_fn = nn.CrossEntropyLoss(ignore_index=pad_token_id)\n    loss_history: list[float] = []\n    total_steps = len(train_loader) * epochs\n\n    with make_progress_bar(total=total_steps, desc="SASRec train", unit="batch") as progress:\n        for _epoch_index in range(epochs):\n            batch_losses: list[float] = []\n            for batch in train_loader:\n                sequences = batch["item_sequences"].to(device=device, non_blocking=device.type == "cuda")\n                labels = batch["labels"].to(device=device, non_blocking=device.type == "cuda")\n\n                model.train()\n                optimizer.zero_grad()\n                logits = model.score_all_items(sequences)\n                logits[:, pad_token_id] = torch.finfo(logits.dtype).min\n                loss = loss_fn(logits, labels)\n                loss.backward()\n                optimizer.step()\n\n                loss_value = float(loss.detach().cpu().item())\n                batch_losses.append(loss_value)\n                progress.set_postfix({"loss": f"{loss_value:.4f}"})\n                progress.update(1)\n            loss_history.append(mean(batch_losses) if batch_losses else 0.0)\n\n    return loss_history\n\n\ndef _rank_sasrec_candidates(\n    *,\n    model: SASRecModel,\n    example: LeaveOneOutExample,\n    candidate_item_ids: list[str],\n    item_to_index: dict[str, int],\n    index_to_item: dict[int, str],\n    max_sequence_length: int,\n    device: torch.device,\n    top_k: int,\n    pad_token_id: int,\n) -> tuple[list[str], float]:\n    """Rank candidate items for one leave-one-out query."""\n\n    history_indices = [item_to_index[item_id] for item_id in example.history_item_ids if item_id in item_to_index]\n    input_ids = _left_pad(history_indices, max_length=max_sequence_length, pad_token_id=pad_token_id)\n    candidate_indices = [item_to_index[item_id] for item_id in candidate_item_ids if item_id in item_to_index]\n    if not candidate_indices:\n        return [], 0.0\n\n    started_at = time.perf_counter()\n    sequence_tensor = torch.tensor([input_ids], dtype=torch.long, device=device)\n    with torch.no_grad():\n        scores = model.score_all_items(sequence_tensor)[0]\n        scores[pad_token_id] = torch.finfo(scores.dtype).min\n        candidate_tensor = torch.tensor(candidate_indices, dtype=torch.long, device=device)\n        candidate_scores = scores.index_select(0, candidate_tensor)\n        sorted_positions = torch.argsort(candidate_scores, descending=True)\n        top_positions = sorted_positions[: min(top_k, sorted_positions.numel())].detach().cpu().tolist()\n    latency_ms = (time.perf_counter() - started_at) * 1000.0\n\n    ranked_indices = [candidate_indices[position] for position in top_positions]\n    ranked_item_ids = [index_to_item[index] for index in ranked_indices]\n    return ranked_item_ids, latency_ms\n\n\ndef _evaluate_sasrec(\n    *,\n    model: SASRecModel,\n    examples: list[LeaveOneOutExample],\n    candidate_item_ids: list[str],\n    item_to_index: dict[str, int],\n    index_to_item: dict[int, str],\n    max_sequence_length: int,\n    device: torch.device,\n    top_k: int,\n    max_eval_candidates: int | None,\n    seed: int,\n    split_name: str,\n    pad_token_id: int = 0,\n) -> dict[str, object]:\n    """Evaluate SASRec on the same ranking metrics as MASI."""\n\n    if not examples:\n        return {\n            f"hr@{top_k}": 0.0,\n            f"ndcg@{top_k}": 0.0,\n            f"coverage@{top_k}": 0.0,\n            "avg_latency_ms": 0.0,\n            "num_examples": 0,\n            "candidate_item_count": len(candidate_item_ids),\n            "max_eval_candidates": max_eval_candidates,\n        }\n\n    model.eval()\n    ranked_lists: list[list[str]] = []\n    hit_scores: list[float] = []\n    ndcg_scores: list[float] = []\n    latencies: list[float] = []\n\n    with make_progress_bar(total=len(examples), desc=f"Evaluate SASRec {split_name}", unit="user") as progress:\n        for example in examples:\n            active_candidate_item_ids = _candidate_pool_for_example(\n                candidate_item_ids=candidate_item_ids,\n                target_item_id=example.target_item_id,\n                max_eval_candidates=max_eval_candidates,\n                seed=seed,\n                user_id=example.user_id,\n            )\n            ranked_item_ids, latency_ms = _rank_sasrec_candidates(\n                model=model,\n                example=example,\n                candidate_item_ids=active_candidate_item_ids,\n                item_to_index=item_to_index,\n                index_to_item=index_to_item,\n                max_sequence_length=max_sequence_length,\n                device=device,\n                top_k=top_k,\n                pad_token_id=pad_token_id,\n            )\n            ranked_lists.append(ranked_item_ids)\n            hit_scores.append(hit_rate_at_k(ranked_item_ids=ranked_item_ids, target_item_id=example.target_item_id, k=top_k))\n            ndcg_scores.append(ndcg_at_k(ranked_item_ids=ranked_item_ids, target_item_id=example.target_item_id, k=top_k))\n            latencies.append(latency_ms)\n            progress.set_postfix({"lat_ms": f"{latency_ms:.1f}", "candidates": len(active_candidate_item_ids)})\n            progress.update(1)\n\n    return {\n        f"hr@{top_k}": mean(hit_scores),\n        f"ndcg@{top_k}": mean(ndcg_scores),\n        f"coverage@{top_k}": coverage_at_k(\n            ranked_lists=ranked_lists,\n            catalog_item_count=len(candidate_item_ids),\n            k=top_k,\n        ),\n        "avg_latency_ms": mean(latencies),\n        "num_examples": len(examples),\n        "candidate_item_count": len(candidate_item_ids),\n        "max_eval_candidates": max_eval_candidates,\n    }\n\n\ndef run_sasrec_baseline(\n    *,\n    user_histories: dict[str, list[str]],\n    import_summary: dict[str, object],\n    config: dict[str, object],\n    device: torch.device,\n    outputs_root: Path,\n    checkpoint_root: Path | None = None,\n) -> dict[str, object]:\n    """Train and evaluate the SASRec baseline on MASI leave-one-out splits."""\n\n    seed = int(config["seed"])\n    baseline_config = dict(config.get("baseline", {}))\n    pad_token_id = 0\n    top_k = int(baseline_config.get("top_k", 10))\n    max_sequence_length = int(baseline_config.get("max_sequence_length", 50))\n    max_eval_candidates_raw = baseline_config.get("max_eval_candidates")\n    max_eval_candidates = None if max_eval_candidates_raw in (None, 0, "0") else int(max_eval_candidates_raw)\n\n    split = build_leave_one_out_split(\n        user_histories=user_histories,\n        cold_start_ratio=float(baseline_config.get("cold_start_ratio", 0.2)),\n        min_train_history=int(baseline_config.get("min_train_history", 1)),\n        seed=seed,\n        use_cold_start_evaluation=bool(baseline_config.get("use_cold_start_evaluation", True)),\n    )\n    item_to_index, index_to_item = _build_item_index(\n        user_histories=user_histories,\n        warm_examples=split.warm_examples,\n        cold_examples=split.cold_examples,\n    )\n    candidate_item_ids = sorted(item_to_index)\n\n    train_dataset = SASRecTrainingDataset(\n        user_histories=split.train_histories,\n        item_to_index=item_to_index,\n        max_sequence_length=max_sequence_length,\n        pad_token_id=pad_token_id,\n    )\n    train_loader = DataLoader(\n        train_dataset,\n        batch_size=int(baseline_config.get("batch_size", 256)),\n        shuffle=len(train_dataset) > 0,\n        generator=torch.Generator().manual_seed(seed),\n        collate_fn=train_dataset.collate,\n    )\n\n    model = SASRecModel(\n        SASRecConfig(\n            num_items=len(item_to_index) + 1,\n            max_sequence_length=max_sequence_length,\n            hidden_dim=int(baseline_config.get("hidden_dim", 128)),\n            num_heads=int(baseline_config.get("num_heads", 4)),\n            num_layers=int(baseline_config.get("num_layers", 3)),\n            dropout=float(baseline_config.get("dropout", 0.1)),\n            pad_token_id=pad_token_id,\n        )\n    )\n    loss_history = _train_sasrec(\n        model=model,\n        train_loader=train_loader,\n        learning_rate=float(baseline_config.get("learning_rate", 0.001)),\n        epochs=int(baseline_config.get("epochs", 30)),\n        device=device,\n        pad_token_id=pad_token_id,\n    )\n\n    warm_metrics = _evaluate_sasrec(\n        model=model,\n        examples=split.warm_examples,\n        candidate_item_ids=candidate_item_ids,\n        item_to_index=item_to_index,\n        index_to_item=index_to_item,\n        max_sequence_length=max_sequence_length,\n        device=device,\n        top_k=top_k,\n        max_eval_candidates=max_eval_candidates,\n        seed=seed,\n        split_name="warm",\n        pad_token_id=pad_token_id,\n    )\n    cold_metrics = _evaluate_sasrec(\n        model=model,\n        examples=split.cold_examples,\n        candidate_item_ids=candidate_item_ids,\n        item_to_index=item_to_index,\n        index_to_item=index_to_item,\n        max_sequence_length=max_sequence_length,\n        device=device,\n        top_k=top_k,\n        max_eval_candidates=max_eval_candidates,\n        seed=seed,\n        split_name="cold",\n        pad_token_id=pad_token_id,\n    )\n\n    outputs_root = ensure_directory(outputs_root)\n    item_mapping_path = write_json(\n        {"item_to_index": item_to_index},\n        outputs_root / "sasrec_item_mapping.json",\n    )\n    checkpoint_paths: dict[str, str] = {}\n    if checkpoint_root is not None:\n        checkpoint_root = ensure_directory(checkpoint_root)\n        model_path = checkpoint_root / "sasrec_model.pt"\n        torch.save(\n            {\n                "config": config,\n                "model_state_dict": module_state_dict_to_cpu(model),\n                "item_to_index": item_to_index,\n                "loss_history": loss_history,\n            },\n            model_path,\n        )\n        checkpoint_paths["sasrec_model"] = str(model_path)\n\n    summary = {\n        "baseline": "sasrec",\n        "baseline_type": "exact_sequential_id_baseline",\n        "seed": seed,\n        "device": str(device),\n        "num_items": len(item_to_index),\n        "num_train_examples": len(train_dataset),\n        "training_status": "trained" if loss_history else "skipped_no_examples_or_epochs",\n        "loss_history": loss_history,\n        "split_summary": split.summary,\n        "warm_metrics": warm_metrics,\n        "cold_metrics": cold_metrics,\n        "import_summary": import_summary,\n        "item_mapping_path": str(item_mapping_path),\n        "checkpoint_paths": checkpoint_paths,\n        "baseline_config": baseline_config,\n    }\n    write_json(summary, outputs_root / "baseline_summary.json")\n    return summary\n',
}

created_files = []
for relative_path, contents in BASELINE_BOOTSTRAP_FILES.items():
    target_path = REPO_DIR / relative_path
    if not target_path.exists():
        target_path.parent.mkdir(parents=True, exist_ok=True)
        target_path.write_text(contents, encoding="utf-8")
        created_files.append(str(target_path.relative_to(REPO_DIR)))

if created_files:
    print("Created missing SASRec baseline artifacts in cloned checkout:")
    print("\n".join(f"- {path}" for path in created_files))
else:
    print("SASRec baseline artifacts already present in checkout.")


In [ ]:
if RUN_PIP_INSTALL:
    pip_base = [sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check"]
    subprocess.run([*pip_base, "--upgrade", "pip"], check=True)
    subprocess.run([*pip_base, "numpy>=1.26,<2.1"], check=True)
    subprocess.run([*pip_base, "-e", ".[recommender]"], check=True)
    print("Packages ready.")
else:
    print("Skipping package installation.")

In [ ]:
def find_prepared_reviews() -> Path | None:
    if not RUNNING_ON_KAGGLE:
        local = REPO_DIR / "data" / "full_dataset" / "Clothing_Shoes_and_Jewelry.jsonl"
        return local if local.exists() else None
    candidates = [
        KAGGLE_INPUT_ROOT / "datasets" / "dheerajrajanala" / "masi-amazon-csj-full-dataset" / "Clothing_Shoes_and_Jewelry.jsonl",
        KAGGLE_INPUT_ROOT / "masi-amazon-csj-full-dataset" / "Clothing_Shoes_and_Jewelry.jsonl",
    ]
    candidates.extend(KAGGLE_INPUT_ROOT.rglob("Clothing_Shoes_and_Jewelry.jsonl"))
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    return None


profile = PROFILES[BASELINE_PROFILE]
base_config = json.loads((REPO_DIR / "configs" / "baseline_sasrec_full_dataset.json").read_text(encoding="utf-8"))
config = deepcopy(base_config)
config["dataset"].update(profile["dataset"])
config["baseline"].update(profile["baseline"])

reviews_path = find_prepared_reviews()
if reviews_path is None:
    raise FileNotFoundError("Could not find Clothing_Shoes_and_Jewelry.jsonl. Attach the prepared MASI CSJ dataset or prepare data/full_dataset locally.")
config["dataset"]["reviews_path"] = str(reviews_path)
run_root = STORAGE_ROOT / "outputs" / config["runtime"]["run_name"]
config["outputs_root"] = str(run_root)
config["checkpoint_root"] = str(run_root / "checkpoints")

resolved_config_dir = STORAGE_ROOT / "resolved_configs"
resolved_config_dir.mkdir(parents=True, exist_ok=True)
resolved_config_path = resolved_config_dir / "baseline_sasrec_full_dataset.json"
resolved_config_path.write_text(json.dumps(config, indent=2), encoding="utf-8")
print(f"Using reviews: {reviews_path}")
print(f"Resolved config: {resolved_config_path}")

In [ ]:
env = dict(os.environ)
env["PYTHONPATH"] = str(REPO_DIR / "src") + os.pathsep + env.get("PYTHONPATH", "")
subprocess.run(
    [sys.executable, str(REPO_DIR / "scripts" / "run_sasrec_baseline.py"), "--config", str(resolved_config_path), "--storage-root", str(STORAGE_ROOT)],
    check=True,
    cwd=REPO_DIR,
    env=env,
)

In [ ]:
summary_path = Path(config["outputs_root"]) / "baseline_summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8"))
print(f"Summary: {summary_path}")
print(json.dumps({
    "baseline": summary["baseline"],
    "num_items": summary["num_items"],
    "num_train_examples": summary["num_train_examples"],
    "warm_metrics": summary["warm_metrics"],
    "cold_metrics": summary["cold_metrics"],
}, indent=2))